# 1. Explore the ticket data
Goal: understand what one row means, check data quality, and look at the daily rhythm of boardings.

Each ticket row says: *how many passengers (`pax`), on which bus (`vehicle`), which `route`, where they boarded (`start_name`), where they are going (`end_name`), and when (`ts`).*
There is no trip id and no stop id, which is the main difficulty of this project.

In [ ]:
import sys, os
sys.path.append("..")                       # lets the notebook import config.py and the crowd/ folder
# os.environ["DATA_MODE"] = "synthetic"     # <- uncomment to try everything on the demo data
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import config
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
P = config.PROC; print("mode:", config.MODE, "| processed folder:", P)

In [ ]:
from crowd import ingest
df = ingest.load_tickets()       # reads data/raw/tickets.csv with pandas
print(df.shape)
df.head()

In [ ]:
df.info()   # check dtypes and missing values

## Data quality profile
`ingest.profile` computes the checks from our plan: nulls, duplicates, date range, daily totals, outlier days, free tickets.

In [ ]:
prof = ingest.profile(df)
print(f"rows: {prof['n_rows']:,}  | range: {prof['date_range'][0]} to {prof['date_range'][1]}")
print('missing start:', prof['missing_start'], '| missing end:', prof['missing_end'], '| duplicate rows:', prof['duplicates'])
prof['ticket_types']

**Question to answer for the technical note:** are tickets with `amount == 0` (free rides) still issued as tickets? If the share of zero amount is high for some `ttype`, these are probably free-ride tickets. They still count as passengers on board.

In [ ]:
daily = prof['daily']
fig, ax = plt.subplots(figsize=(10, 3.5))
sns.lineplot(data=daily, x='date', y='pax', marker='o', ax=ax)
sns.scatterplot(data=daily[daily.outlier], x='date', y='pax', color='red', s=120, ax=ax, label='outlier weekday')
ax.tick_params(axis='x', rotation=90); ax.set_title('Passengers per day'); plt.show()

In [ ]:
df['daytype'] = np.where(df.dow < 6, 'weekday', 'weekend')
hourly = df.groupby(['daytype', 'hour']).pax.sum().div(df.groupby('daytype').date.nunique(), level='daytype').rename('avg_pax').reset_index()
sns.lineplot(data=hourly, x='hour', y='avg_pax', hue='daytype', marker='o'); plt.title('Average passengers by hour'); plt.show()

You should see morning and evening peaks on weekdays and a flatter weekend. If not, something is wrong with the time column.

In [ ]:
df.to_parquet(P / 'tickets.parquet')      # saved for the next notebook
print('saved')